# MAA — Analisis S2, S8, S10 (tanpa training) — v2: jarak S8 diperbaiki (kuantil 5–95 %, skala IQR)

- **S2** kualitas pose per peserta dan per jenis alat bantu: confidence, sendi hilang (lengan/kaki), kaki tersilang.
- **S8** realisme augmentasi: proporsi tubuh dan gerak hasil U1–U4 (dari peserta tanpa alat bantu) dibanding skeleton asli
  pengguna kursi roda, kruk, tongkat. Data P07–P13 hanya dipakai di sini sebagai pembanding, tidak untuk latih atau tuning.
- **S10** biaya: parameter, GFLOPs, waktu per klip 64 frame di GPU dan CPU 1 thread.

Cukup **satu sesi**, runtime T4 (GPU hanya untuk S10), **Run all**. ±10 menit. Hasil: `MyDrive/CARR/maa/results/` dan `maa/figures/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA = '/content/drive/MyDrive/CARR'

In [ ]:
import os
os.environ['CARR_REPO'] = '/content/CARR'
!git clone -q https://github.com/januaradip/CARR.git /content/CARR 2>/dev/null; git -C /content/CARR fetch -q
!git -C /content/CARR checkout -q 1193346
!pip install -q thop

In [ ]:
%%writefile /content/maa.py
"""
MAA: Mobility-Aid-Aware skeleton augmentation for activity recognition on CARR.

Experiment S1 (this version): baselines without the proposed augmentation (aug 'A0') for four backbones
(ST-GCN, ST-GCN++, CTR-GCN, PoseC3D) on the nine CARR protocols (CS1-4, CL1-4, CG) and several seeds.

Builds on the public CARR benchmark code (github.com/januaradip/CARR):
  * frame index, protocol splits, run locks, trainer with checkpoint/resume  -> carr.common
  * 17 COCO keypoints (YOLOv8m-Pose), box-normalised sequences, ST-GCN / CTR-GCN -> carr.skeleton

Layout of the output (WORK):
  WORK/cache/frames.pkl, WORK/cache/seq_box.pkl         shared caches
  WORK/maa/runs/<exp>/<aug>/<model>/<split>/s<seed>/    checkpoints, DONE.json, video_probs.npz
  WORK/maa/results/<exp>_runs.csv, <exp>_summary.csv    collected results

Every unit (exp, aug, model, split, seed) is claimed with a lock file, so several Colab sessions (also from
different Google accounts sharing the same Drive folder) can work on the same experiment in parallel.
"""
import os, sys, glob, json, math, time, random, shutil, collections
import numpy as np

CARR_REPO = os.environ.get('CARR_REPO', '/content/CARR')
if CARR_REPO not in sys.path:
    sys.path.insert(0, CARR_REPO)
from carr import common as C                   # noqa: E402
from carr import skeleton as SK                # noqa: E402

MODELS = ['ST-GCN', 'ST-GCN++', 'CTR-GCN', 'PoseC3D']
SPLITS = ['CS1', 'CS2', 'CS3', 'CS4', 'CL1', 'CL2', 'CL3', 'CL4', 'CG']

CFG = dict(
    WIN=64, STRIDE=32,                         # windows as in the CARR benchmark
    EPOCHS=60, PATIENCE=12, WARMUP=3, WD=1e-4, WORKERS=2,
    BATCH={'ST-GCN': 64, 'ST-GCN++': 64, 'CTR-GCN': 64, 'PoseC3D': 32},
    ACCUM={'ST-GCN': 1, 'ST-GCN++': 1, 'CTR-GCN': 1, 'PoseC3D': 2},     # PoseC3D: effective batch 64
    LR={'ST-GCN': 1e-3, 'ST-GCN++': 1e-3, 'CTR-GCN': 1e-3, 'PoseC3D': 1e-3},
    HM_SIZE=56, HM_SIGMA=0.6, HM_RANGE=0.75, HM_FRAMES=32,              # PoseC3D heatmap volumes
)


# =============================================================================== data
def paths(data, work):
    return C.Paths(data, work)


def load_seq(P, F, pose_dir=None):
    """Box-normalised keypoint sequences {video: (T,17,3)}, cached once in WORK/cache/seq_box.pkl.
    pose_dir: folder with <video>.npz from the CARR benchmark (default WORK/cache/pose)."""
    cache = f'{P.cache}/seq_box.pkl'
    if os.path.exists(cache):
        obj = C.load_pickle(cache)
        if obj.get('sig') == P.signature() and len(obj['seq']) == len(F['videos']):
            return obj['seq']
    pose_dir = pose_dir or f'{P.cache}/pose'
    missing = [v for v in F['videos'] if not os.path.exists(f'{pose_dir}/{v}.npz')]
    if missing:
        raise FileNotFoundError(
            f'{len(missing)} of {len(F["videos"])} pose files missing in {pose_dir} (e.g. {missing[:3]}). '
            f'Set POSE_DIR to the pose cache of the CARR benchmark (CARR_work/cache/pose) or run the '
            f'pose-extraction cell.')
    C.log('Building box-normalised sequences (once)...')
    # carr.skeleton.load_sequences reads <cache>/pose/<video>.npz
    if os.path.basename(pose_dir.rstrip('/')) != 'pose':
        raise ValueError('POSE_DIR must be a folder named "pose" (e.g. .../CARR_work/cache/pose)')
    q = type('Q', (), {})()
    q.cache = os.path.dirname(pose_dir.rstrip('/'))
    seq = SK.load_sequences(q, F)
    C.save_pickle(dict(sig=P.signature(), seq=seq), cache)
    C.log(f'{len(seq)} sequences cached in {cache}')
    return seq


def make_windows(videos, seq):
    return SK.make_windows(videos, seq, None)


# ------------------------------------------------------------------ augmentations
def base_aug(x):
    """Augmentation of the CARR benchmark (kept for every variant): flip, rotation +-10 deg, scale 0.9-1.1."""
    if random.random() < 0.5:
        x = x[:, SK.FLIP]
        x[:, :, 0] *= -1
    a = math.radians(random.uniform(-10, 10))
    sc = random.uniform(0.9, 1.1)
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], np.float32) * sc
    x[:, :, :2] = x[:, :, :2] @ R.T
    return x


try:
    import augs as _AG                         # generic (A1-A4) and proposed (U, U1-U4, U5) augmentations
    AUGS = dict(_AG.AUGS)
except ImportError:
    _AG, AUGS = None, {'A0': lambda x: x}


def parse_aug(name):
    """'U+U5' -> (augmentation 'U', upper-body normalisation on); 'U5' alone -> ('A0', True)."""
    parts = name.split('+')
    norm = 'U5' in parts
    rest = '+'.join(p for p in parts if p != 'U5') or 'A0'
    return rest, norm


class SkelDS:
    def __init__(self, items, seq, F, train, aug='A0'):
        name, self.norm = parse_aug(aug)
        self.items, self.seq, self.F, self.train, self.aug = items, seq, F, train, AUGS[name]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        import torch
        v, idx = self.items[i]
        x = self.seq[v][idx].copy()                    # (T,17,3): x, y (box-normalised), confidence
        if self.train:
            x = self.aug(base_aug(x))
        if self.norm:
            x = _AG.u5_upper_norm(x)
        x = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).permute(2, 0, 1).contiguous()
        return x, self.F['videos'][v]['cls'], i


# =============================================================================== models
def build(name):
    import torch
    import torch.nn as nn
    if name in ('ST-GCN', 'CTR-GCN'):
        return SK.build_model(name)

    A0 = torch.tensor(SK.graph_A())                   # (3,17,17)

    class DataBN(nn.Module):
        def __init__(s, C=3, V=17):
            super().__init__()
            s.bn = nn.BatchNorm1d(C * V)

        def forward(s, x):
            N, Cc, T, V = x.shape
            x = x.permute(0, 3, 1, 2).reshape(N, V * Cc, T)
            return s.bn(x).reshape(N, V, Cc, T).permute(0, 2, 3, 1).contiguous()

    # ------------------------------------------------ ST-GCN++ (PYSKL: adaptive 'init' graph + multi-scale TCN)
    class AGCN(nn.Module):
        def __init__(s, cin, cout):
            super().__init__()
            s.K = A0.shape[0]
            s.A = nn.Parameter(A0.clone())             # initialised with the skeleton graph, fully learnable
            s.conv = nn.Conv2d(cin, cout * s.K, 1)
            s.bn = nn.BatchNorm2d(cout)
            s.res = nn.Identity() if cin == cout else nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            N, _, T, V = x.shape
            y = s.conv(x).view(N, s.K, -1, T, V)
            y = torch.einsum('nkctv,kwv->nctw', y, s.A)
            return s.relu(s.bn(y) + s.res(x))

    class MSTCN(nn.Module):
        def __init__(s, cin, cout, stride=1):
            super().__init__()
            nb = 6
            mids = [cout // nb] * (nb - 1) + [cout - (cout // nb) * (nb - 1)]
            br = []
            for d, m in zip((1, 2, 3, 4), mids[:4]):
                br.append(nn.Sequential(nn.Conv2d(cin, m, 1), nn.BatchNorm2d(m), nn.ReLU(inplace=True),
                                        nn.Conv2d(m, m, (3, 1), (stride, 1), (d, 0), (d, 1))))
            br.append(nn.Sequential(nn.Conv2d(cin, mids[4], 1), nn.BatchNorm2d(mids[4]), nn.ReLU(inplace=True),
                                    nn.MaxPool2d((3, 1), (stride, 1), (1, 0))))
            br.append(nn.Conv2d(cin, mids[5], 1, (stride, 1)))
            s.br = nn.ModuleList(br)
            s.out = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.Conv2d(cout, cout, 1),
                                  nn.BatchNorm2d(cout))

        def forward(s, x):
            return s.out(torch.cat([b(x) for b in s.br], 1))

    class PPBlock(nn.Module):
        def __init__(s, cin, cout, stride=1, residual=True):
            super().__init__()
            s.gcn, s.tcn = AGCN(cin, cout), MSTCN(cout, cout, stride)
            if not residual:
                s.res = lambda x: 0
            elif cin == cout and stride == 1:
                s.res = nn.Identity()
            else:
                s.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            return s.relu(s.tcn(s.gcn(x)) + s.res(x))

    class STGCNpp(nn.Module):
        def __init__(s):
            super().__init__()
            s.bn = DataBN()
            cfg = [(3, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True), (64, 64, 1, True),
                   (64, 128, 2, True), (128, 128, 1, True), (128, 128, 1, True),
                   (128, 256, 2, True), (256, 256, 1, True), (256, 256, 1, True)]
            s.blocks = nn.ModuleList(PPBlock(*c) for c in cfg)
            s.drop = nn.Dropout(0.1)
            s.fc = nn.Linear(256, 6)

        def forward(s, x):
            x = s.bn(x)
            for b in s.blocks:
                x = b(x)
            return s.fc(s.drop(x.mean((2, 3))))

    # ------------------------------------------------ PoseC3D (PYSKL: SlowOnly-R50 pose variant on joint heatmaps)
    class Bottleneck3d(nn.Module):
        def __init__(s, cin, planes, stride_s=1, inflate=False):
            super().__init__()
            kt = 3 if inflate else 1
            s.c1 = nn.Sequential(nn.Conv3d(cin, planes, (kt, 1, 1), padding=(kt // 2, 0, 0), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c2 = nn.Sequential(nn.Conv3d(planes, planes, (1, 3, 3), (1, stride_s, stride_s), (0, 1, 1), bias=False),
                                 nn.BatchNorm3d(planes), nn.ReLU(inplace=True))
            s.c3 = nn.Sequential(nn.Conv3d(planes, planes * 4, 1, bias=False), nn.BatchNorm3d(planes * 4))
            s.down = None
            if stride_s != 1 or cin != planes * 4:
                s.down = nn.Sequential(nn.Conv3d(cin, planes * 4, 1, (1, stride_s, stride_s), bias=False),
                                       nn.BatchNorm3d(planes * 4))
            s.relu = nn.ReLU(inplace=True)

        def forward(s, x):
            r = x if s.down is None else s.down(x)
            return s.relu(s.c3(s.c2(s.c1(x))) + r)

    class PoseC3D(nn.Module):
        """Input (N,3,T,17) box-normalised keypoints -> joint heatmap volume (N,17,T/2,56,56) built on the GPU."""

        def __init__(s):
            super().__init__()
            S = CFG['HM_SIZE']
            g = torch.linspace(-CFG['HM_RANGE'], CFG['HM_RANGE'], S)
            s.register_buffer('gx', g.view(1, 1, 1, 1, S))
            s.register_buffer('gy', g.view(1, 1, 1, S, 1))
            s.sig = CFG['HM_SIGMA'] * (2 * CFG['HM_RANGE'] / (S - 1))     # sigma in grid cells -> coordinates
            s.stem = nn.Sequential(nn.Conv3d(17, 32, (1, 7, 7), padding=(0, 3, 3), bias=False),
                                   nn.BatchNorm3d(32), nn.ReLU(inplace=True))
            layers, cin = [], 32
            for planes, n, inflate in ((32, 4, False), (64, 6, True), (128, 3, True)):
                for i in range(n):
                    layers.append(Bottleneck3d(cin, planes, 2 if i == 0 else 1, inflate))
                    cin = planes * 4
            s.body = nn.Sequential(*layers)
            s.drop = nn.Dropout(0.5)
            s.fc = nn.Linear(cin, 6)

        def heatmaps(s, x):
            T = x.shape[2]
            step = max(1, T // CFG['HM_FRAMES'])
            x = x[:, :, ::step][:, :, :CFG['HM_FRAMES']].float()       # (N,3,T',17)
            u = x[:, 0].permute(0, 2, 1)[..., None, None]               # (N,17,T',1,1)
            v = x[:, 1].permute(0, 2, 1)[..., None, None]
            c = x[:, 2].permute(0, 2, 1)[..., None, None]
            return c * torch.exp(-((s.gx - u) ** 2 + (s.gy - v) ** 2) / (2 * s.sig ** 2))

        def forward(s, x):
            h = s.heatmaps(x)
            return s.fc(s.drop(s.body(s.stem(h)).mean((2, 3, 4))))

    return {'ST-GCN++': STGCNpp, 'PoseC3D': PoseC3D}[name]()


def n_params(m):
    return sum(p.numel() for p in m.parameters())


# =============================================================================== units
def units(exp, augs, models, splits, seeds):
    return [dict(exp=exp, aug=a, model=m, split=s, seed=int(sd))
            for a in augs for m in models for s in splits for sd in seeds]


def unit_dir(P, u):
    return f"{P.work}/maa/runs/{u['exp']}/{u['aug']}/{u['model']}/{u['split']}/s{u['seed']}"


def unit_tag(u):
    return f"{u['exp']} {u['aug']} {u['model']} {u['split']} s{u['seed']}"


def evaluate(rdir, model, test_ds, F, cfg):
    """Video-level macro-F1 (mean of window probabilities per video), overall and per group; saves the
    per-video probabilities for the paired bootstrap."""
    import torch
    from torch.utils.data import DataLoader
    model.load_state_dict(torch.load(f'{rdir}/best.pt', map_location='cpu', weights_only=False)['model'])
    model.to('cuda')
    dl = DataLoader(test_ds, batch_size=cfg['batch'], shuffle=False, num_workers=cfg['workers'], pin_memory=True)
    yt, yp, probs, idx = C.predict(model, dl, 'cuda', True)
    wv = np.array([test_ds.items[i][0] for i in idx])
    vids = sorted(set(wv))
    vp = np.stack([probs[wv == v].mean(0) for v in vids])
    vy = np.array([F['videos'][v]['cls'] for v in vids])
    grp = np.array([F['videos'][v]['group'] for v in vids])
    own = np.array([F['videos'][v]['owner'] for v in vids])
    np.savez_compressed(f'{rdir}/video_probs.npz', videos=np.array(vids), y=vy, probs=vp, group=grp, owner=own)
    res = {}
    for tag, m in (('all', np.ones(len(vids), bool)), ('dis', grp == 'dis'), ('nondis', grp == 'nondis')):
        if m.sum():
            r = C.classification_metrics(vy[m], vp[m].argmax(1))
            res[f'vid_f1_{tag}'], res[f'vid_acc_{tag}'], res[f'n_{tag}'] = r['macro_f1'], r['acc'], int(m.sum())
            if tag == 'all':
                C.save_json(r['cm'], f'{rdir}/confusion_video.json')
    r = C.classification_metrics(yt, yp)
    res['win_f1_all'], res['win_acc_all'] = r['macro_f1'], r['acc']
    return res


def run_unit(P, F, seq, u):
    import torch
    rdir = unit_dir(P, u)
    C.seed_everything(u['seed'])
    torch.backends.cudnn.benchmark = True
    parts = C.load_split(P, u['split'])
    tr, va, te = (make_windows(parts[k], seq) for k in ('train', 'val', 'test'))
    m = u['model']
    cfg = dict(epochs=CFG['EPOCHS'], lr=CFG['LR'][m], wd=CFG['WD'], batch=CFG['BATCH'][m], accum=CFG['ACCUM'][m],
               patience=CFG['PATIENCE'], workers=CFG['WORKERS'], warmup=CFG['WARMUP'], amp=True)
    C.log(f'windows train {len(tr)} / val {len(va)} / test {len(te)}')
    t0 = time.time()
    C.train_classifier(rdir, build(m), SkelDS(tr, seq, F, True, u['aug']), SkelDS(va, seq, F, False, u['aug']), cfg)
    res = evaluate(rdir, build(m), SkelDS(te, seq, F, False, u['aug']), F, cfg)
    hist = json.load(open(f'{rdir}/history.json')) if os.path.exists(f'{rdir}/history.json') else []
    res.update(u, params=n_params(build(m)), epochs=len(hist),
               best_val_f1=max([h['val_f1'] for h in hist], default=None),
               gpu=torch.cuda.get_device_name(0), minutes=round((time.time() - t0) / 60, 1))
    C.save_json(res, f'{rdir}/DONE.json')
    C.log(f"{unit_tag(u)} finished: video macro-F1 {res['vid_f1_all']:.4f}"
          + (f" (dis {res['vid_f1_dis']:.4f})" if 'vid_f1_dis' in res else ''))


def execute(P, F, seq, todo, order='forward', session_seed=None):
    """todo: list of units, or list of stages (lists of units); stages are done in order, shuffled within."""
    stages = todo if todo and isinstance(todo[0], list) else [todo]
    rnd = random.Random(session_seed if session_seed is not None else C.SESSION)
    todo = []
    for st in stages:
        st = list(st)
        if order == 'reverse':
            st = st[::-1]
        elif order == 'shuffle':
            rnd.shuffle(st)
        todo += st
    busy = []
    for u in todo:
        rdir = unit_dir(P, u)
        if os.path.exists(f'{rdir}/DONE.json'):
            continue
        if not C.claim_run(rdir):
            busy.append(unit_tag(u))
            continue
        C.log(f'===== {unit_tag(u)} =====')
        try:
            run_unit(P, F, seq, u)
        finally:
            C.release_run(rdir)
    left = [u for u in todo if not os.path.exists(f'{unit_dir(P, u)}/DONE.json')]
    C.log(f'Session finished. Open units: {len(left)} (in progress elsewhere: {len(busy)}).')
    return left


def status(P, todo):
    import pandas as pd
    rows = []
    for u in todo:
        d = unit_dir(P, u)
        st = 'done' if os.path.exists(f'{d}/DONE.json') else ('running' if os.path.exists(f'{d}/LOCK') else
                                                              ('started' if os.path.exists(f'{d}/history.json') else 'open'))
        rows.append(dict(u, status=st))
    df = pd.DataFrame(rows)
    return df.pivot_table(index=['aug', 'model'], columns='status', values='seed', aggfunc='count', fill_value=0)


# =============================================================================== results
def collect(P, exp):
    """exp: name or list of names (e.g. ['S1', 'S3'] so that the A0 baselines of S1 are included)."""
    import pandas as pd
    exps = [exp] if isinstance(exp, str) else list(exp)
    rows = []
    for e in exps:
        for f in glob.glob(f'{P.work}/maa/runs/{e}/*/*/*/*/DONE.json'):
            with open(f) as fh:
                rows.append(json.load(fh))
    if not rows:
        C.log('No finished units yet.')
        return None
    df = pd.DataFrame(rows)
    os.makedirs(f'{P.work}/maa/results', exist_ok=True)
    df.to_csv(f'{P.work}/maa/results/{"_".join(exps)}_runs.csv', index=False)
    return df


def summary(P, exp):
    """Video macro-F1 (%) per model and protocol: CS (mean of 4 folds), CS-dis, CL, CG; mean +- sd over seeds."""
    import pandas as pd
    df = collect(P, exp)
    if df is None:
        return None
    df['protocol'] = df['split'].str.replace(r'\d+$', '', regex=True)
    out = []
    for (aug, model), g in df.groupby(['aug', 'model']):
        row = dict(aug=aug, model=model)
        for prot, col, name in (('CS', 'vid_f1_all', 'CS'), ('CS', 'vid_f1_dis', 'CS-dis'),
                                ('CL', 'vid_f1_all', 'CL'), ('CG', 'vid_f1_all', 'CG')):
            h = g[g.protocol == prot]
            if h.empty or col not in h:
                continue
            per_seed = h.groupby('seed')[col].mean() * 100          # fold mean per seed
            nf = h.groupby('seed')['split'].nunique()
            full = nf.max() == (1 if prot == 'CG' else 4)
            row[name] = f"{per_seed.mean():.1f} ± {per_seed.std(ddof=1) if len(per_seed) > 1 else 0:.1f}" \
                        f" (n={len(per_seed)}{'' if full else ', incomplete'})"
        row['params_M'] = round(g['params'].iloc[0] / 1e6, 2)
        row['min_per_run'] = round(g['minutes'].median(), 1)
        out.append(row)
    s = pd.DataFrame(out)
    tag = exp if isinstance(exp, str) else '_'.join(exp)
    s.to_csv(f'{P.work}/maa/results/{tag}_summary.csv', index=False)
    return s


# ------------------------------------------------------------------ paired comparison with bootstrap CI
def _f1(y, p, k=6):
    f = []
    for c in range(k):
        if (y == c).any():
            tp = np.sum((y == c) & (p == c)); fp = np.sum((y != c) & (p == c)); fn = np.sum((y == c) & (p != c))
            f.append(2 * tp / max(1, 2 * tp + fp + fn))
    return float(np.mean(f)) if f else 0.0


def _pooled(P, exp, aug, model, splits, seed):
    """Video predictions of one seed pooled over the folds of a protocol (each video is tested once)."""
    out = {}
    for sp in splits:
        f = f"{P.work}/maa/runs/{exp}/{aug}/{model}/{sp}/s{seed}/video_probs.npz"
        if not os.path.exists(f):
            return None
        z = np.load(f, allow_pickle=True)
        for v, y, p, g in zip(z['videos'], z['y'], z['probs'], z['group']):
            out[str(v)] = (int(y), int(np.argmax(p)), str(g))
    return out


def compare(P, variants, models, seeds=(0, 1, 2, 3, 4), base=('S1', 'A0'), n_boot=2000):
    """Macro-F1 difference variant - base (points), mean over the seeds both have, with a paired bootstrap over
    videos (95 % CI). Rows: CG, CS (pooled over 4 folds) all / dis / nondis, CL dis (if available)."""
    import pandas as pd
    rng = np.random.default_rng(0)
    prots = {'CG': ['CG'], 'CS': ['CS1', 'CS2', 'CS3', 'CS4'], 'CL': ['CL1', 'CL2', 'CL3', 'CL4']}
    rows = []
    for (vexp, vaug) in variants:
        for m in models:
            for prot, sps in prots.items():
                pairs = []
                for sd in seeds:
                    b, v = _pooled(P, base[0], base[1], m, sps, sd), _pooled(P, vexp, vaug, m, sps, sd)
                    if b and v:
                        pairs.append((b, v))
                if not pairs:
                    continue
                vids = sorted(pairs[0][0])
                grp = np.array([pairs[0][0][k][2] for k in vids])
                for sub in (['all', 'dis', 'nondis'] if prot != 'CG' else ['all']):
                    if prot == 'CL' and sub != 'dis':
                        continue
                    idx = np.arange(len(vids)) if sub == 'all' else np.where(grp == sub)[0]
                    if len(idx) == 0:
                        continue
                    Y = np.array([[pr[0][vids[i]][0] for i in idx] for pr in pairs])
                    PB = np.array([[pr[0][vids[i]][1] for i in idx] for pr in pairs])
                    PV = np.array([[pr[1][vids[i]][1] for i in idx] for pr in pairs])
                    fb = np.mean([_f1(Y[k], PB[k]) for k in range(len(pairs))])
                    fv = np.mean([_f1(Y[k], PV[k]) for k in range(len(pairs))])
                    d = []
                    for _ in range(n_boot):
                        r = rng.integers(0, len(idx), len(idx))
                        d.append(np.mean([_f1(Y[k, r], PV[k, r]) - _f1(Y[k, r], PB[k, r]) for k in range(len(pairs))]))
                    lo, hi = np.percentile(d, [2.5, 97.5]) * 100
                    rows.append(dict(variant=f'{vexp}:{vaug}', model=m, protocol=prot if sub == 'all' else f'{prot}-{sub}',
                                     seeds=len(pairs), videos=len(idx), base=round(fb * 100, 1), variant_f1=round(fv * 100, 1),
                                     diff=round((fv - fb) * 100, 1), ci_lo=round(lo, 1), ci_hi=round(hi, 1),
                                     significant=bool(lo > 0 or hi < 0)))
    df = pd.DataFrame(rows)
    if len(df):
        os.makedirs(f'{P.work}/maa/results', exist_ok=True)
        df.to_csv(f'{P.work}/maa/results/compare_vs_{base[0]}_{base[1]}.csv', index=False)
    return df


In [ ]:
%%writefile /content/augs.py
"""
Skeleton augmentations for MAA (COCO-17, box-normalised: x right, y down, box centre = 0, box height = 1).
Input/output of every transform: float32 array (T, 17, 3) = (x, y, confidence); confidence 0 = missing joint.

Generic (comparison):
  A1  strong affine: rotation +-20 deg, scale 0.8-1.2, shear +-0.2, translation +-0.1
  A2  joint dropout (10 % of joints) and body-part dropout (one of arms / legs / head, p = 0.3)
  A3  temporal: speed perturbation 0.7-1.3 with resampling to the same length
  A4  A1 + A2 + A3

Proposed mobility-aid kinematic augmentations (applied to one sample with probability P_AID):
  U1  seated (wheelchair): static seated lower body, upper body lowered by the thigh length
  U2  occupied arm (single crutch / cane, as in CARR): one arm fixed beside the body
  U3  asymmetric gait: reduced motion of one leg, slower tempo, lateral trunk sway
  U4  aid occlusion: lower legs (footrest) or aid-side forearm missing, or noisy / swapped legs, over a time segment
  U   one of U1-U4 drawn at random (U1 may be combined with U4-legs)
  U5  upper-body-centred normalisation (a representation, used at training and test time)
After U1-U3 the skeleton is re-fitted to the original keypoint extent, as the annotated box would be.
"""
import math, random
import numpy as np

NOSE, LSHO, RSHO, LELB, RELB, LWRI, RWRI, LHIP, RHIP, LKNE, RKNE, LANK, RANK = 0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16
UPPER = list(range(0, 13))              # head, arms, hips
LEGS = [LKNE, RKNE, LANK, RANK]
FLIP = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
P_AID = 0.5


# ----------------------------------------------------------------------------- helpers
def _valid(x):
    return x[..., 2] > 0


def _mean_pt(x, j):
    """Mean position of joint j over the valid frames (or None)."""
    v = _valid(x[:, j])
    return x[v, j, :2].mean(0) if v.any() else None


def _seg_len(x, a, b):
    v = _valid(x[:, a]) & _valid(x[:, b])
    return float(np.linalg.norm(x[v, a, :2] - x[v, b, :2], axis=1).mean()) if v.any() else None


def _extent(x):
    v = _valid(x)
    if not v.any():
        return None
    p = x[v][:, :2]
    lo, hi = p.min(0), p.max(0)
    return (lo + hi) / 2, max(hi[1] - lo[1], 1e-3)


def refit(x_new, x_old, full=None):
    """Re-centre on the original keypoint extent and re-scale towards it. The box of a seated person or of a
    person with an aid is drawn around the person *and* the aid, so the skeleton fills the box only partly:
    the scale is drawn between 'unchanged' (u = 0) and 'fills the original extent' (u = 1)."""
    e0, e1 = _extent(x_old), _extent(x_new)
    if e0 is None or e1 is None:
        return x_new
    (c0, h0), (c1, h1) = e0, e1
    u = random.uniform(0, 1) if full is None else float(full)
    k = (h0 / h1) ** u
    v = _valid(x_new)
    x_new[..., :2] = np.where(v[..., None], (x_new[..., :2] - c1) * k + c0, 0)
    return x_new


def _resample(x, factor):
    """Play the sequence `factor` times faster (factor < 1: slower) and bring it back to length T."""
    T = len(x)
    src = np.clip(np.arange(T) * factor, 0, T - 1)
    if factor < 1:                                  # slower: random start so that all phases occur
        src = src + random.uniform(0, max(0.0, (T - 1) - src[-1]))
    i0 = np.floor(src).astype(int)
    i1 = np.minimum(i0 + 1, T - 1)
    w = (src - i0)[:, None, None]
    out = x[i0] * (1 - w) + x[i1] * w
    both = _valid(x[i0]) & _valid(x[i1])
    out[..., 2] = np.where(both, out[..., 2], 0)
    out[..., :2] = np.where(both[..., None], out[..., :2], 0)
    return out.astype(np.float32)


def _drop(x, joints, t0=0, t1=None):
    x[t0:t1, joints] = 0
    return x


# ----------------------------------------------------------------------------- generic
def a1_affine(x):
    a = math.radians(random.uniform(-20, 20))
    s = random.uniform(0.8, 1.2)
    sh = random.uniform(-0.2, 0.2)
    M = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]], np.float32) @ \
        np.array([[1, sh], [0, 1]], np.float32) * s
    t = np.random.uniform(-0.1, 0.1, 2).astype(np.float32)
    v = _valid(x)
    x[..., :2] = np.where(v[..., None], x[..., :2] @ M.T + t, 0)
    return x


def a2_dropout(x):
    m = np.random.rand(17) < 0.1
    x[:, m] = 0
    if random.random() < 0.3:
        part = random.choice([[LELB, LWRI, RELB, RWRI], LEGS, [0, 1, 2, 3, 4]])
        x[:, part] = 0
    return x


def a3_speed(x):
    return _resample(x, random.uniform(0.7, 1.3))


def a4_generic(x):
    return a2_dropout(a1_affine(a3_speed(x)))


# ----------------------------------------------------------------------------- proposed
def u1_seated(x):
    """Seated in a wheelchair, seen from an elevated camera: torso lowered and slightly leaning, thighs pointing
    towards the camera (short in the image) with the knees apart, shanks foreshortened with the feet on the
    footrest, legs static. The person fills only part of the box (the box also contains the chair)."""
    old = x.copy()
    lth = _seg_len(x, LHIP, LKNE) or _seg_len(x, RHIP, RKNE) or 0.22
    lsh = _seg_len(x, LKNE, LANK) or _seg_len(x, RKNE, RANK) or 0.22
    sho_c = (x[:, LSHO, :2] + x[:, RSHO, :2]) / 2
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    # 1. lean: shoulders and head come closer to the hips
    lean = random.uniform(0.2, 0.45)                # seated torso looks shorter (lean, slumped, seen from above)
    top = [0, 1, 2, 3, 4, LSHO, RSHO, LELB, RELB, LWRI, RWRI]
    x[:, top, 1] -= np.where(_valid(x[:, top]), lean * (sho_c[:, None, 1] - hip_c[:, None, 1]), 0)
    # 2. the whole upper body comes down by about one thigh length
    drop = lth * random.uniform(0.6, 1.0)
    x[:, UPPER, 1] += np.where(_valid(x[:, UPPER]), drop, 0)
    # 3. seated legs (identical shape over time, they move with the chair)
    f = random.uniform(0.15, 0.55)                  # thigh seen almost end-on from the elevated camera
    spread = random.uniform(0.02, 0.07)             # knees apart
    shank = lsh * random.uniform(0.5, 0.8)          # feet forward on the footrest
    inward = random.uniform(-0.01, 0.03)
    hw = np.abs(x[:, LHIP, 0] - x[:, RHIP, 0]).mean() / 2 + 1e-6
    for hip, kne, ank in ((LHIP, LKNE, LANK), (RHIP, RKNE, RANK)):
        hv = _valid(x[:, hip])
        if not hv.any():
            x[:, [kne, ank]] = 0
            continue
        out = np.sign((x[hv, hip, 0] - hip_c[hv, 0]).mean()) or 1.0
        kn = x[:, hip, :2] + np.array([out * spread, lth * f], np.float32)
        an = kn + np.array([-out * inward, shank], np.float32)
        jit = lambda: np.random.normal(0, 0.003, (len(x), 2)).astype(np.float32)
        x[:, kne, :2], x[:, ank, :2] = kn + jit(), an + jit()
        x[:, kne, 2] = np.where(hv, x[:, hip, 2], 0)
        x[:, ank, 2] = np.where(hv, x[:, hip, 2] * 0.8, 0)
        x[~hv, kne, :2] = 0
        x[~hv, ank, :2] = 0
    del hw
    return refit(x, old, full=random.uniform(0.0, 0.2))


def u2_arm(x):
    old = x.copy()
    sides = [(LSHO, LELB, LWRI, LHIP), (RSHO, RELB, RWRI, RHIP)]
    random.shuffle(sides)
    sides = sides[:1]                              # CARR: crutch and cane users hold a single aid
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    out_w = random.uniform(0.05, 0.15)
    down_w = random.uniform(0.0, 0.12)
    for k, (sho, elb, wri, hip) in enumerate(sides):
        ok = _valid(x[:, sho]) & _valid(x[:, hip])
        if not ok.any():
            continue
        # outward = away from the body centre on the image (depends on whether the person faces the camera)
        outward = 1.0 if (x[ok, hip, 0] - hip_c[ok, 0]).mean() >= 0 else -1.0
        w = x[:, hip, :2] + np.array([outward * out_w, down_w], np.float32)
        e = (x[:, sho, :2] + w) / 2 + np.array([outward * 0.03, 0], np.float32)
        noise = lambda: np.random.normal(0, 0.004, (len(x), 2)).astype(np.float32)
        x[ok, wri, :2], x[ok, elb, :2] = (w + noise())[ok], (e + noise())[ok]
        x[ok, wri, 2] = x[ok, elb, 2] = np.minimum(x[ok, sho, 2], x[ok, hip, 2])
    return refit(x, old)


def u3_gait(x):
    old = x.copy()
    x = _resample(x, random.uniform(0.6, 0.85))           # slower tempo
    hip_c = (x[:, LHIP, :2] + x[:, RHIP, :2]) / 2
    side = random.choice([(LKNE, LANK, LHIP), (RKNE, RANK, RHIP)])
    a = random.uniform(0.3, 0.8)                           # remaining motion of the affected leg
    for j in side[:2]:
        v = _valid(x[:, j]) & _valid(x[:, side[2]])
        if v.any():
            rel = x[:, j, :2] - x[:, side[2], :2]
            m = rel[v].mean(0)
            x[v, j, :2] = x[v, side[2], :2] + m + (rel[v] - m) * a
    amp = random.uniform(0.015, 0.045)                     # lateral trunk sway with each step
    per = random.uniform(15, 30)
    ph = random.uniform(0, 2 * math.pi)
    sway = amp * np.sin(2 * math.pi * np.arange(len(x)) / per + ph)
    up = [0, 1, 2, 3, 4, LSHO, RSHO, LELB, RELB, LWRI, RWRI]
    x[:, up, 0] += np.where(_valid(x[:, up]), sway[:, None], 0)
    return refit(x, old)


def u4_occlusion(x, legs=None):
    T = len(x)
    if legs is None:
        legs = random.random() < 0.5
    if random.random() < 0.5:
        t0, t1 = 0, T
    else:
        L = random.randint(T // 4, T)
        t0 = random.randint(0, T - L)
        t1 = t0 + L
    if legs:
        if random.random() < 0.4:                  # aid confuses the pose estimator: noisy, crossed legs
            v = _valid(x[t0:t1, LEGS])
            x[t0:t1, LEGS, :2] += np.where(v[..., None], np.random.normal(0, 0.03, (t1 - t0, 4, 2)), 0).astype(np.float32)
            if random.random() < 0.5:
                x[t0:t1, [LKNE, RKNE, LANK, RANK]] = x[t0:t1, [RKNE, LKNE, RANK, LANK]]
            return x
        return _drop(x, random.choice([[LANK, RANK], LEGS]), t0, t1)
    return _drop(x, random.choice([[LWRI], [RWRI], [LELB, LWRI], [RELB, RWRI]]), t0, t1)


def u_any(x, which=('U1', 'U2', 'U3', 'U4')):
    if random.random() >= P_AID:
        return x
    k = random.choice(which)
    if k == 'U1':
        x = u1_seated(x)
        if random.random() < 0.3:
            x = u4_occlusion(x, legs=True)
        return x
    return {'U2': u2_arm, 'U3': u3_gait, 'U4': u4_occlusion}[k](x)


def u5_upper_norm(x):
    """Upper-body-centred normalisation of a whole window: origin = mid-shoulder of the first valid frame,
    unit = median shoulder width x 3 (about the box height of a standing adult). Falls back to the input."""
    v = _valid(x[:, LSHO]) & _valid(x[:, RSHO])
    if v.sum() < 2:
        return x
    mid = (x[v, LSHO, :2] + x[v, RSHO, :2]) / 2
    sw = np.median(np.linalg.norm(x[v, LSHO, :2] - x[v, RSHO, :2], axis=1))
    if sw < 0.03:
        return x
    o, s = mid[0], 1.0 / (3 * sw)
    vv = _valid(x)
    x[..., :2] = np.where(vv[..., None], (x[..., :2] - o) * s, 0)
    return x


def single(name):
    return {'U1': u1_seated, 'U2': u2_arm, 'U3': u3_gait, 'U4': u4_occlusion}[name]


AUGS = {
    'A0': lambda x: x,
    'A1': a1_affine, 'A2': a2_dropout, 'A3': a3_speed, 'A4': a4_generic,
    'U1': lambda x: u_any(x, ('U1',)), 'U2': lambda x: u_any(x, ('U2',)),
    'U3': lambda x: u_any(x, ('U3',)), 'U4': lambda x: u_any(x, ('U4',)),
    'U': u_any,
    'A4+U': lambda x: u_any(a4_generic(x)),
    'A4+U1': lambda x: u_any(a4_generic(x), ('U1',)),
    'U1+U4': lambda x: u_any(x, ('U1', 'U4')),
}


In [ ]:
%%writefile /content/maa_analysis.py
"""
MAA analyses without training:
  S2  pose-quality diagnosis per participant (keypoint confidence, missing joints, leg crossings, proportions)
  S8  realism of the augmentations: body-proportion and motion features of augmented skeletons (from participants
      without aids) versus real skeletons of aid users (P07-P13 are only used here, never for training or tuning)
  S10 cost: parameters, GFLOPs per 64-frame clip, latency per clip (GPU and 1 CPU thread)
Output: <DATA>/maa/results/S2_*.csv, S8_*.csv, S10_cost.csv and figures in <DATA>/maa/figures/.
"""
import os, math, random, time
import numpy as np
import augs as A

AID = {'P07': 'wheelchair', 'P11': 'wheelchair', 'P08': 'crutches', 'P10': 'crutches', 'P13': 'crutches',
       'P09': 'cane', 'P12': 'cane'}
LS, RS, LH, RH, LK, RK, LA, RA = 5, 6, 11, 12, 13, 14, 15, 16
ARMS, LEGS = [7, 8, 9, 10], [13, 14, 15, 16]


def group_of(owner):
    return AID.get(owner, 'no aid')


# ----------------------------------------------------------------------------- per-window features
def _mid(x, a, b):
    v = (x[:, a, 2] > 0) & (x[:, b, 2] > 0)
    m = (x[:, a, :2] + x[:, b, :2]) / 2
    return m, v


def features(x):
    """Body-proportion and motion features of one window (T,17,3); NaN where not computable.
    Lengths are divided by the shoulder width, which does not change between standing and sitting."""
    f = {}
    sho, vs = _mid(x, LS, RS)
    hip, vh = _mid(x, LH, RH)
    ank, va = _mid(x, LA, RA)
    kne, vk = _mid(x, LK, RK)
    sw = np.linalg.norm(x[:, LS, :2] - x[:, RS, :2], axis=1)
    ok = vs & (x[:, LS, 2] > 0) & (x[:, RS, 2] > 0) & (sw > 0.02)
    swm = np.median(sw[ok]) if ok.any() else np.nan

    def med(a, m):
        return float(np.median(a[m])) if m.any() else np.nan
    f['torso / shoulder'] = med(hip[:, 1] - sho[:, 1], vs & vh) / swm
    f['hip-knee / shoulder'] = med(kne[:, 1] - hip[:, 1], vh & vk) / swm
    f['knee-ankle / shoulder'] = med(ank[:, 1] - kne[:, 1], vk & va) / swm
    f['shoulder-ankle / shoulder'] = med(ank[:, 1] - sho[:, 1], vs & va) / swm
    f['torso / shoulder-ankle'] = f['torso / shoulder'] / f['shoulder-ankle / shoulder'] \
        if f['shoulder-ankle / shoulder'] and not np.isnan(f['shoulder-ankle / shoulder']) else np.nan
    # motion: leg swing relative to the hips, hand reach relative to the shoulders (per shoulder width)
    leg = []
    for j, h in ((LA, LH), (RA, RH), (LK, LH), (RK, RH)):
        v = (x[:, j, 2] > 0) & (x[:, h, 2] > 0)
        if v.sum() > 4:
            leg.append(np.std(x[v, j, 1] - x[v, h, 1]))
    f['leg swing / shoulder'] = (np.mean(leg) / swm) if leg else np.nan
    # arm use: motion of each wrist relative to its shoulder; an arm holding an aid moves little
    arm = []
    for w, sh in ((9, LS), (10, RS)):
        v = (x[:, w, 2] > 0) & (x[:, sh, 2] > 0)
        arm.append(np.linalg.norm(np.std(x[v, w, :2] - x[v, sh, :2], axis=0)) / swm if v.sum() > 4 else np.nan)
    f['quieter arm activity'] = float(np.nanmin(arm)) if not np.all(np.isnan(arm)) else np.nan
    f['arm activity ratio'] = float(np.nanmin(arm) / max(np.nanmax(arm), 1e-6)) if not np.any(np.isnan(arm)) else np.nan
    ls = (x[:, LA, 2] > 0) & (x[:, RA, 2] > 0) & (x[:, LK, 2] > 0) & (x[:, RK, 2] > 0)
    f['leg asymmetry'] = abs(np.std(x[ls, LA, 1] - x[ls, LK, 1]) - np.std(x[ls, RA, 1] - x[ls, RK, 1])) / swm \
        if ls.sum() > 4 else np.nan
    return f


def quality(x):
    """Pose-quality indicators of one window."""
    c = x[..., 2]
    q = dict(conf_all=float(c[c > 0].mean()) if (c > 0).any() else 0.0,
             missing_all=float((c <= 0).mean()),
             missing_arms=float((c[:, ARMS] <= 0).mean()), missing_legs=float((c[:, LEGS] <= 0).mean()),
             lowconf_legs=float((c[:, LEGS] < 0.5).mean()))
    # left/right ankles swapped relative to the hips (crossed legs) in frames with all four joints
    v = (c[:, LA] > 0) & (c[:, RA] > 0) & (c[:, LH] > 0) & (c[:, RH] > 0)
    if v.any():
        hs = np.sign(x[v, LH, 0] - x[v, RH, 0])
        as_ = np.sign(x[v, LA, 0] - x[v, RA, 0])
        q['crossed_legs'] = float((hs != as_).mean())
    else:
        q['crossed_legs'] = np.nan
    return q


# ----------------------------------------------------------------------------- S2
def s2_quality(seq, F, out):
    import pandas as pd
    rows = []
    for v, x in seq.items():
        m = F['videos'][v]
        if len(x) < 8:
            continue
        r = quality(x)
        r.update(video=v, participant=m['owner'], aid=group_of(m['owner']), cls=m['cls'])
        rows.append(r)
    df = pd.DataFrame(rows)
    cols = ['conf_all', 'missing_all', 'missing_arms', 'missing_legs', 'lowconf_legs', 'crossed_legs']
    by_p = df.groupby(['aid', 'participant'])[cols].mean().round(3)
    by_g = df.groupby('aid')[cols].mean().round(3)
    by_p.to_csv(f'{out}/S2_pose_quality_participant.csv')
    by_g.to_csv(f'{out}/S2_pose_quality_group.csv')
    return by_g, by_p


# ----------------------------------------------------------------------------- S8
def _windows(seq, vids, T=64, per_video=4, rng=None):
    rng = rng or np.random.default_rng(0)
    out = []
    for v in vids:
        x = seq[v]
        if len(x) < T:
            out.append(np.concatenate([x, np.zeros((T - len(x), 17, 3), np.float32)]))
            continue
        for s in rng.choice(len(x) - T + 1, min(per_video, len(x) - T + 1), replace=False):
            out.append(x[s:s + T].copy())
    return out


def _wasserstein(a, b):
    """1-D Wasserstein distance on the 5-95 % quantile range (ratios explode when a denominator is ~0, e.g.
    side views with a tiny shoulder width; the tails would otherwise dominate)."""
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 5 or len(b) < 5:
        return np.nan
    q = np.linspace(0.05, 0.95, 91)
    return float(np.mean(np.abs(np.quantile(a, q) - np.quantile(b, q))))


def s8_realism(seq, F, out, figdir, n_per_video=4, seed=0):
    """Compare features of: real windows without aids, the same windows after U1 / U2 / U3 / U4 / U,
    and real windows of aid users per aid type. Distance = 1-D Wasserstein per feature, scaled by the
    spread of the real no-aid feature; lower = closer to the real aid users."""
    import pandas as pd
    random.seed(seed); np.random.seed(seed)
    rng = np.random.default_rng(seed)
    vids = {g: [v for v in seq if group_of(F['videos'][v]['owner']) == g] for g in
            ('no aid', 'wheelchair', 'crutches', 'cane')}
    base = _windows(seq, vids['no aid'], per_video=n_per_video, rng=rng)
    sets = {'real: no aid': base}
    for g in ('wheelchair', 'crutches', 'cane'):
        sets[f'real: {g}'] = _windows(seq, vids[g], per_video=n_per_video, rng=rng)
    aug = {'U1 seated': A.u1_seated, 'U2 occupied arm': A.u2_arm, 'U3 gait': A.u3_gait,
           'U4 occlusion': A.u4_occlusion, 'A4 generic': A.a4_generic}
    for k, fn in aug.items():
        sets[f'aug: {k}'] = [fn(w.copy()) for w in base]
    feats = {k: pd.DataFrame([features(w) for w in ws]) for k, ws in sets.items()}
    names = list(feats['real: no aid'].columns)
    rows = []
    for k, df in feats.items():
        r = {'set': k, 'windows': len(df)}
        r.update({n: round(float(np.nanmedian(df[n])), 3) for n in names})
        rows.append(r)
    med = pd.DataFrame(rows)
    med.to_csv(f'{out}/S8_feature_medians.csv', index=False)
    # distances to each real aid group
    pairs = {'wheelchair': ['real: no aid', 'aug: U1 seated', 'aug: A4 generic'],
             'crutches': ['real: no aid', 'aug: U2 occupied arm', 'aug: U3 gait', 'aug: U4 occlusion', 'aug: A4 generic'],
             'cane': ['real: no aid', 'aug: U3 gait', 'aug: U2 occupied arm', 'aug: A4 generic']}
    drows = []
    for g, srcs in pairs.items():
        tgt = feats[f'real: {g}']
        for s in srcs:
            r = {'target': g, 'source': s}
            ds = []
            for n in names:
                ref = feats['real: no aid'][n].values
                ref = ref[np.isfinite(ref)]
                sd = (np.percentile(ref, 75) - np.percentile(ref, 25)) or 1.0      # robust spread (IQR)
                d = _wasserstein(feats[s][n].values, tgt[n].values) / sd
                r[n] = round(d, 3)
                if not np.isnan(d):
                    ds.append(d)
            r['mean'] = round(float(np.mean(ds)), 3) if ds else np.nan
            drows.append(r)
    dist = pd.DataFrame(drows)
    dist.to_csv(f'{out}/S8_distance_to_real_aid_users.csv', index=False)
    _plot_s8(feats, figdir)
    return med, dist


def _plot_s8(feats, figdir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    show = ['torso / shoulder', 'hip-knee / shoulder', 'shoulder-ankle / shoulder', 'leg swing / shoulder']
    groups = [('real: no aid', '#888888', '-'), ('real: wheelchair', '#c0392b', '-'), ('aug: U1 seated', '#c0392b', '--'),
              ('real: crutches', '#2471a3', '-'), ('aug: U3 gait', '#2471a3', '--')]
    fig, axs = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.4))
    for ax, n in zip(axs, show):
        allv = np.concatenate([feats[g][n].dropna().values for g, _, _ in groups])
        lo, hi = np.nanpercentile(allv, [1, 99])
        bins = np.linspace(lo, hi, 40)
        for g, col, ls in groups:
            v = feats[g][n].dropna().values
            if len(v) > 5:
                h, e = np.histogram(np.clip(v, lo, hi), bins=bins, density=True)
                ax.plot((e[1:] + e[:-1]) / 2, h, color=col, ls=ls, lw=1.6, label=g)
        ax.set_title(n, fontsize=10)
        ax.set_yticks([])
    axs[0].legend(fontsize=7, frameon=False)
    plt.tight_layout()
    os.makedirs(figdir, exist_ok=True)
    plt.savefig(f'{figdir}/S8_feature_distributions.png', dpi=150)
    plt.close(fig)


# ----------------------------------------------------------------------------- S10
def s10_cost(build, models, out, reps=30):
    import torch
    import pandas as pd
    rows = []
    x1 = torch.randn(1, 3, 64, 17) * 0.3
    x1[:, 2] = torch.rand(1, 64, 17)
    for m in models:
        net = build(m).eval()
        r = dict(model=m, params_M=round(sum(p.numel() for p in net.parameters()) / 1e6, 2))
        try:
            from thop import profile
            macs, _ = profile(net, inputs=(x1,), verbose=False)
            r['GFLOPs_per_clip'] = round(2 * macs / 1e9, 3)
        except Exception as e:                       # thop missing or op not supported
            r['GFLOPs_per_clip'] = np.nan
        with torch.no_grad():
            torch.set_num_threads(1)
            for _ in range(3):
                net(x1)
            t = time.perf_counter()
            for _ in range(reps):
                net(x1)
            r['cpu1_ms_per_clip'] = round((time.perf_counter() - t) / reps * 1000, 2)
            torch.set_num_threads(os.cpu_count() or 2)
            if torch.cuda.is_available():
                g = net.cuda()
                xg = x1.cuda()
                for _ in range(10):
                    g(xg)
                torch.cuda.synchronize()
                t = time.perf_counter()
                for _ in range(reps * 3):
                    g(xg)
                torch.cuda.synchronize()
                r['gpu_ms_per_clip'] = round((time.perf_counter() - t) / (reps * 3) * 1000, 2)
        r['ms_per_frame_cpu1'] = round(r['cpu1_ms_per_clip'] / 64, 3)
        rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(f'{out}/S10_cost.csv', index=False)
    return df


In [ ]:
import sys, importlib, pandas as pd
sys.path.insert(0, '/content')
import maa, augs, maa_analysis as MA
for m in (augs, maa, MA): importlib.reload(m)
from carr import common as C
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
P = maa.paths(DATA, DATA)
F = C.load_frames(P)
seq = maa.load_seq(P, F, f'{P.cache}/pose')
OUT, FIG = f'{DATA}/maa/results', f'{DATA}/maa/figures'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

## S2 — kualitas pose

In [ ]:
g, p = MA.s2_quality(seq, F, OUT)
display(g); display(p)

## S8 — realisme augmentasi
`S8_distance_to_real_aid_users`: jarak (Wasserstein, dinormalisasi) ke skeleton asli tiap kelompok alat bantu; makin kecil makin mirip. Bandingkan baris augmentasi dengan baris `real: no aid`.

In [ ]:
med, dist = MA.s8_realism(seq, F, OUT, FIG)
display(med); display(dist)
from IPython.display import Image
display(Image(f'{FIG}/S8_feature_distributions.png'))

## S10 — biaya

In [ ]:
display(MA.s10_cost(maa.build, maa.MODELS, OUT))
print('Selesai. File di', OUT)